In [0]:
# ============================================================
# NEXPULSE — PIPELINE AUDIT / OBSERVABILITY HELPER
# ============================================================

import uuid
from datetime import datetime, timezone

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StringType,
    TimestampType,
    LongType,
    DoubleType
)


# ============================================================
# AUDIT SCHEMA
# ============================================================

audit_schema = (
    StructType()
    .add("pipeline_run_id", StringType())
    .add("pipeline_name", StringType())
    .add("source", StringType())
    .add("start_time", TimestampType())
    .add("end_time", TimestampType())
    .add("records_read", LongType())
    .add("records_written", LongType())
    .add("records_quarantined", LongType())
    .add("records_deduplicated", LongType())
    .add("status", StringType())
    .add("error_message", StringType())
    .add("processing_latency_seconds", DoubleType())
)


# ============================================================
# AUDIT DELTA TABLE PATH
# ============================================================

audit_table_path = (
    "abfss://gold@adlsnexpulse01.dfs.core.windows.net/"
    "audit/pipeline_runs/"
)


# ============================================================
# LOG PIPELINE RUN
# ============================================================

def log_pipeline_run(
    pipeline_name,
    source,
    start_time,
    records_read=0,
    records_written=0,
    records_quarantined=0,
    records_deduplicated=0,
    status="success",
    error_message=None
):

    end_time = datetime.now(timezone.utc)

    latency = (
        end_time - start_time
    ).total_seconds()

    row = Row(
        pipeline_run_id=str(uuid.uuid4()),
        pipeline_name=pipeline_name,
        source=source,
        start_time=start_time,
        end_time=end_time,
        records_read=records_read,
        records_written=records_written,
        records_quarantined=records_quarantined,
        records_deduplicated=records_deduplicated,
        status=status,
        error_message=error_message,
        processing_latency_seconds=latency
    )

    audit_df = spark.createDataFrame(
        [row],
        schema=audit_schema
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .save(audit_table_path)
    )

    print(
        f"[audit] logged run "
        f"{row.pipeline_run_id} "
        f"({status}), "
        f"latency={latency:.1f}s"
    )

In [0]:
from datetime import datetime, timezone

test_start = datetime.now(timezone.utc)

log_pipeline_run(
    pipeline_name="audit_helper_test",
    source="test",
    start_time=test_start,
    records_read=1,
    records_written=1,
    records_quarantined=0,
    records_deduplicated=0,
    status="success"
)

[audit] logged run cb24b51a-43ca-48aa-8379-89d3340c939f (success), latency=0.0s
